In [0]:
%python
%pip install --upgrade \
    "mlflow>=2.20,<3.0" \
    "langchain>=0.1.16,<0.2.0" \
    "databricks-vectorsearch>=0.40" \
    "databricks-sdk>=0.40"

dbutils.library.restartPython()

Looking in indexes: [REDACTED]
INFO: pip is looking at multiple versions of databricks-vectorsearch to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 145.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 621.4/621.4 kB 38.9 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.0.0
    Not uninstalling t

In [0]:
%python
import os
host="https://adb-xxxxxxxxx.azuredatabricks.net"
os.environ["DATABRICKS_TOKEN"] = "xxxxxxxxx"
VECTOR_SEARCH_ENDPOINT_NAME="vector_endpoint",
index_name="Vector_Index_Table"

                 User Question
                       │
                       ▼
              "What is Databricks?"
                       │
                       ▼
                Embedding Model
                       │
                       ▼
                 Query Vector
                       │
                       ▼
          Databricks Vector Search
                       │
                       ▼
              Similarity Search
                       │
             ┌─────────┼─────────┐
             ▼         ▼         ▼
          Doc #1     Doc #2    Doc #3
             │         │         │
             └─────────┼─────────┘
                       ▼
                 LangChain
                  Retriever
                       │
                       ▼
              Relevant documents
                       │
                       ▼
                      LLM
                       │
                       ▼
                 Final answer

In [0]:
%python

from databricks.vector_search.client import VectorSearchClient
from langchain_community.vectorstores import DatabricksVectorSearch


def get_retriever(_=None):
    # Use no arguments - auto-authenticates via service principal in serving
    vsc = VectorSearchClient()

    vs_index = vsc.get_index(
        endpoint_name="vector_endpoint",
        index_name="Vector_Index_Table
    )

    # No embedding param needed - index uses Databricks-managed embeddings
    vectorstore = DatabricksVectorSearch(
        vs_index,
        text_column="doc_text"
    )

    return vectorstore.as_retriever()

-----%python
from databricks.vector_search.client import VectorSearchClient
from langchain_community.vectorstores import DatabricksVectorSearch
from langchain_community.embeddings import DatabricksEmbeddings
from langchain.schema import Document



embeddings = DatabricksEmbeddings(endpoint="databricks-gte-large-en")
def get_retriever(persist_dir: str =""):
    os.environ["DATABRICKS_HOST"] = "https://adb-1643293829894738.18.azuredatabricks.net"
    #get vector search index
    vsc = VectorSearchClient(workspace_url=host, personal_access_token=os.environ["DATABRICKS_TOKEN"])
    vs_index = vsc.get_index(
    endpoint_name="vector_endpoint",
    index_name="vneteswar.default.index_on_deltatable"
)
#Create a retriever
    vectorstore =DatabricksVectorSearch(vs_index,
    embedding=embeddings,
    text_column="doc_text",
    )
----    return vectorstore.as_retriever()

In [0]:
#not required 
os.environ["DATABRICKS_HOST"] = "https://adb-xxxxxxxxxxx.azuredatabricks.net"
    #get vector search index
vsc = VectorSearchClient(workspace_url=host, personal_access_token=os.environ["DATABRICKS_TOKEN"])
vsc.list_indexes("vector_endpoint")

In [0]:
%python
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_community.chat_models import ChatDatabricks
chat_model=ChatDatabricks(endpoint = "databricks-meta-llama-3-1-8b-instruct",max_tokens=200)
Template ="""You are an assistant for Azure Databricks support tickets. Provide troubleshooting steps and solutions for the user's query.

Rules:
- Do NOT include any preamble like "Based on the provided context" or "I'll answer the question".
- Jump straight to the answer. Start directly with the resolution or troubleshooting steps.
- If the question is not related to Azure Databricks, politely decline.
- If you don't know the answer, say so. Do not make up answers.
- If you don't have data on the topic, say so.
- Keep the answer concise and in English only.

Context:
{context}

Question: {question}

Answer:
"""
prompt=PromptTemplate(template=Template,input_variables=["context","question"])
chain=RetrievalQA.from_chain_type(llm=chat_model,chain_type="stuff",retriever=get_retriever(),chain_type_kwargs={"prompt":prompt})


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


In [0]:
%python
question={"query":"PowerBI OAuth2.0 authentication to Databricks"}
answer=chain.run(question)


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


In [0]:
%python
import builtins
print = builtins.print

In [0]:
%python

question = {"query": "PowerBI OAuth2.0 authentication to Databricks"}

answer = chain.invoke(question)

# Print only the result text, not the full dict
print(answer["result"])

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
To troubleshoot the PowerBI OAuth2.0 authentication to Databricks issue, please follow these steps:

1. Review the client secret configuration: Ensure that the client secret is being passed from the application via App Registrations. If so, create a client secret in the Databricks portal for the associated service principal and use that value instead.
2. Verify the token scope: Ensure that the token scope in the connection has "offline_access" to avoid the "403 Forbidden" error.
3. Check the service principal configuration: Verify that two separate applications are created in Entra ID - one to represent the Snowflake resource and another to be used for the connector from Databricks.
4. Review the connections: Check if the connection is correctly configured and if the catalog is s

In [0]:
    -- %python
    -- from mlflow.models import infer_signature
    -- import mlflow
    -- import langchain
    -- mlflow.set_registry_uri("databricks-uc")
    -- model_name="vneteswar.default.spticketchatbot_model_New"
    -- with mlflow.start_run(run_name="spticketchatbot_model") as run:
    --     signature =infer_signature(question,answer)
    --     model_info=mlflow.langchain.log_model(chain,
    --                                         loader_fn=get_retriever,
    --                                         artifact_path="chain",
    --                                         registered_model_name=model_name,
    --                                         pip_requirements=[
    --                                             "mlflow==" + mlflow.__version__,
    --                                             "langchain==" + langchain.__version__,
    --                                             "databricks-vectorsearch"
    --                                         ],
    --                                         input_example=question,
    --                                         signature=signature)

2026/08/23 23:11:18 INFO mlflow: Attempting to auto-detect Databricks resource dependencies for the current langchain model. Dependency auto-detection is best-effort and may not capture all dependencies of your langchain model, resulting in authorization errors when serving or querying your model. We recommend that you explicitly pass `resources` to mlflow.langchain.log_model() to ensure authorization to dependent resources succeeds when the model is deployed.
2026/08/23 23:11:18 WARNING mlflow.models.model: Failed to validate serving input example {
  "inputs": {
    "query": "PowerBI OAuth2.0 authentication to Databricks"
  }
}. Alternatively, you can avoid passing input example and pass model signature instead when logging the model. To ensure the input example is valid prior to serving, please try calling `mlflow.models.validate_serving_input` on the model uri and serving input example. A serving input example can be generated from model input example using `mlflow.models.convert_i

Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

Registered model 'vneteswar.default.spticketchatbot_model_New' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

Created version '2' of model 'vneteswar.default.spticketchatbot_model_new'.


In [0]:
%python
from databricks.vector_search.client import VectorSearchClient

host = os.environ["DATABRICKS_HOST"]
token = os.environ["DATABRICKS_TOKEN"]

vsc = VectorSearchClient(
    workspace_url=host,
    personal_access_token=token
)

index = vsc.get_index(
    endpoint_name="vector_endpoint",
    index_name="vneteswar.default.index_on_deltatable"
)

print("SUCCESS - Vector Search authentication works")

[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
SUCCESS - Vector Search authentication works


In [0]:
%python
import os

print("TOKEN:", bool(os.getenv("DATABRICKS_TOKEN")))
print("HOST:", os.getenv("DATABRICKS_HOST"))

from databricks.vector_search.client import VectorSearchClient

print("VectorSearchClient imported successfully")

TOKEN: True
HOST: https://adb-1643293829894738.18.azuredatabricks.net
VectorSearchClient imported successfully


In [0]:
%python
print(os.environ.get("DATABRICKS_TOKEN", "")[:5])

dapi4


In [0]:
%python
vsc = VectorSearchClient(
    workspace_url=os.environ["DATABRICKS_HOST"],
    personal_access_token=os.environ["DATABRICKS_TOKEN"]
)

vs_index = vsc.get_index(
    endpoint_name="vector_endpoint",
    index_name="Vector_index_table"
)

print("VECTOR SEARCH CONNECTION SUCCESS")

[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
VECTOR SEARCH CONNECTION SUCCESS


In [0]:
%python

retriever = get_retriever(None)

print("Retriever created successfully")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Retriever created successfully


In [0]:
%python

docs = retriever.invoke(
    "PowerBI OAuth2.0 authentication to Databricks"
)

print("Documents retrieved:", len(docs))

for doc in docs[:3]:
    print(doc.page_content[:500])
    print("---")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Documents retrieved: 4
ERROR:
PowerBI OAuth2.0 authentication to Databricks

RESOLUTION:
•	Had a discussion with you to review the issue.
•	It was observed that the client secret was being passed from the application via App Registrations.
•	Advised you to create a client secret in the Databricks portal for the associated service principal, and use that value.
•	After implementing the change, the connection worked successfully.
•	Additionally, we located the relevant Machine-to-Machine (M2M) documentation.

o	Authoriz
---
error:the call from the Azure API to the databricks endpoint results in a “403 Forbidden” error, indicating that the user is not authorized..resolution: N/A
---
ERROR:
•	It was reviewed the error with backline.
•	Team provided below details.
•	Point to highlight

In [0]:
%python

from mlflow.models import infer_signature
import mlflow
import langchain

mlflow.set_registry_uri("databricks-uc")

model_name = "vneteswar.default.spticketchatbot_model_New"

question = {
    "query": "PowerBI OAuth2.0 authentication to Databricks"
}

answer = chain.invoke(question)

with mlflow.start_run(run_name="spticketchatbot_model") as run:

    signature = infer_signature(
        question,
        answer
    )

    model_info = mlflow.langchain.log_model(
        chain,
        loader_fn=get_retriever,
        artifact_path="chain",
        registered_model_name=model_name,
        pip_requirements=[
            "mlflow==" + mlflow.__version__,
            "langchain==" + langchain.__version__,
            "databricks-vectorsearch"
        ],
        input_example=question,
        signature=signature
    )

print("Model logged successfully")
print("Model URI:", model_info.model_uri)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-b4922a99-58ee-4403-abdf-8d91ad515f1f/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


2026/08/24 01:18:03 INFO mlflow.pyfunc: Validating input example against model signature


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


2026/08/24 01:18:06 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

Registered model 'vneteswar.default.spticketchatbot_model_New' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

Model logged successfully
Model URI: runs:/c3f2c5da820b425ab41c951bcada1977/chain


Created version '7' of model 'vneteswar.default.spticketchatbot_model_new'.
